# Lab 03: Статистический анализ продаж (NumPy)
Кейс: региональная эффективность интернет-магазина (продажи в тыс. ден. ед. за 8 недель).

In [3]:
import numpy as np

np.set_printoptions(suppress=True, precision=2)

# строки — регионы, столбцы — недели
sales = np.array([
    [120, 125, 130, 128, 135, 140, 138, 145],
    [80, 82, 79, 85, 88, 90, 86, 92],
    [150, 148, 155, 160, 158, 165, 220, 170],
    [60, 65, 62, 58, 64, 63, 61, 66],
    [100, 105, 102, 108, 110, 112, 109, 115]
])

region_names = np.array(["Bishkek", "Osh", "Karakol", "Tokmok", "Jalal-Abad"])


def table(**columns):
    # вспомогательная функция: колонки -> таблица для печати
    return np.rec.fromarrays(list(columns.values()), names=list(columns.keys()))

## Задание 1. Сравнение регионов

In [4]:
total_revenue = sales.sum(axis=1)                # сумма по неделям для каждого региона
mean_weekly_sales = sales.mean(axis=1)
median_weekly_sales = np.median(sales, axis=1)

print(table(region=region_names,
            total=total_revenue,
            mean=np.round(mean_weekly_sales, 2),
            median=median_weekly_sales))

leader_region = region_names[np.argmax(total_revenue)]

# «типичный» регион: медиана региона ближе всего к медиане всех наблюдений
overall_median = np.median(sales)
typical_region = region_names[np.argmin(np.abs(median_weekly_sales - overall_median))]

print("\nЛидер по сумме:", leader_region)
print("Медиана по всем данным:", overall_median)
print("Самый типичный регион:", typical_region)

[('Bishkek', 1061, 132.62, 132.5) ('Osh',  682,  85.25,  85.5)
 ('Karakol', 1326, 165.75, 159. ) ('Tokmok',  499,  62.38,  62.5)
 ('Jalal-Abad',  861, 107.62, 108.5)]

Лидер по сумме: Karakol
Медиана по всем данным: 108.5
Самый типичный регион: Jalal-Abad


**Вывод.** Лидер по сумме — **Karakol** (1326). Типичную выручку лучше описывает **медиана**: она не зависит от всплесков
(у Karakol среднее 165.75 выше медианы 159 из-за недели 220). Медиана Jalal-Abad (108.5) совпадает с медианой всех данных,
поэтому это самый «типичный» регион.

## Задание 2. Анализ разброса

In [3]:
std_weekly = sales.std(axis=1, ddof=1)           # выборочное стд. отклонение (8 недель — выборка)
q1 = np.percentile(sales, 25, axis=1)
q3 = np.percentile(sales, 75, axis=1)
iqr = q3 - q1

# относительный разброс, % (учитывает масштаб продаж)
coef_variation = std_weekly / mean_weekly_sales * 100
iqr_to_median = iqr / median_weekly_sales * 100

print(table(region=region_names,
            std=np.round(std_weekly, 2),
            iqr=iqr,
            cv_percent=np.round(coef_variation, 1),
            iqr_to_median_percent=np.round(iqr_to_median, 1)))

print("\nСамый стабильный (по CV):", region_names[np.argmin(coef_variation)])
print("Самый нестабильный (по CV):", region_names[np.argmax(coef_variation)])

[('Bishkek',  8.35, 11.25,  6.3, 8.5) ('Osh',  4.68,  7.  ,  5.5, 8.2)
 ('Karakol', 23.09, 12.5 , 13.9, 7.9) ('Tokmok',  2.67,  3.5 ,  4.3, 5.6)
 ('Jalal-Abad',  5.04,  6.25,  4.7, 5.8)]

Самый стабильный (по CV): Tokmok
Самый нестабильный (по CV): Karakol


**Вывод.** В абсолютных числах разброс растёт с масштабом, поэтому сравниваем по CV (std / среднее).
Самые стабильные — **Tokmok** (CV 4.3%), **Jalal-Abad** (4.7%), **Osh** (5.5%). Больше всего разброс у **Karakol** (CV 13.9%) —
но по IQR/медиане Karakol (7.9%) не хуже Bishkek (8.5%): основной разброс создаёт один выброс.

## Задание 3. Поиск выбросов (правило IQR)

In [4]:
lower_bound = (q1 - 1.5 * iqr)[:, None]          # [:, None] — столбец, чтобы сравнивать с каждой неделей
upper_bound = (q3 + 1.5 * iqr)[:, None]

outlier_mask = (sales < lower_bound) | (sales > upper_bound)

outlier_region_idx, outlier_week_idx = np.nonzero(outlier_mask)

print(table(region=region_names, outliers_count=outlier_mask.sum(axis=1)))
print(table(region=region_names[outlier_region_idx],
            week=outlier_week_idx + 1,
            value=sales[outlier_mask],
            lower=lower_bound[outlier_region_idx, 0],
            upper=upper_bound[outlier_region_idx, 0]))

[('Bishkek', 0) ('Osh', 0) ('Karakol', 1) ('Tokmok', 0) ('Jalal-Abad', 0)]
[('Karakol', 7, 220, 135., 185.)]


**Единственный выброс: Karakol, неделя 7 = 220** (граница 185; соседние недели 165 и 170, рост ≈ +33%).

- *Реальный всплеск:* акция или распродажа, сезонный пик/праздник, крупный разовый заказ.
- *Ошибка данных:* опечатка (220 вместо 170/120), дубль записи, смешение валют/единиц, перенос продаж другой недели.

Выход за границу ≠ ошибка — значение нужно проверить по источнику.

## Задание 4. Анализ структуры

In [5]:
store_total = total_revenue.sum()
assert store_total != 0                           # проверка знаменателя

revenue_share = total_revenue / store_total * 100

top_region = region_names[np.argmax(revenue_share)]
top_share = revenue_share.max()
top2_share = np.sort(revenue_share)[-2:].sum()
herfindahl_index = np.sum((revenue_share / 100) ** 2)   # при равных долях 5 регионов = 0.2

print(table(region=region_names, share_percent=np.round(revenue_share, 1)))
print("\nКрупнейший регион:", top_region, f"({top_share:.1f}%)")
print(f"Доля двух крупнейших: {top2_share:.1f}%")
print(f"Индекс Херфиндаля: {herfindahl_index:.3f} (равномерное распределение = 0.200)")

[('Bishkek', 24. ) ('Osh', 15.4) ('Karakol', 29.9) ('Tokmok', 11.3)
 ('Jalal-Abad', 19.4)]

Крупнейший регион: Karakol (29.9%)
Доля двух крупнейших: 53.9%
Индекс Херфиндаля: 0.221 (равномерное распределение = 0.200)


**Вывод.** Karakol даёт ≈30% выручки, два крупнейших региона (Karakol + Bishkek) — ≈54%. Индекс Херфиндаля 0.22 ≈ равномерному 0.20:
зависимость от одного региона умеренная, но потеря любого из двух лидеров заметно ударит по выручке.
Доля Karakol частично раздута выбросом (см. задание 6).

## Задание 5. Сравнение периодов

In [6]:
first_half_mean = sales[:, :4].mean(axis=1)       # недели 1–4
last_half_mean = sales[:, 4:].mean(axis=1)        # недели 5–8

absolute_change = last_half_mean - first_half_mean

# процент считаем только там, где база != 0, иначе NaN
percent_change = np.full(first_half_mean.shape, np.nan)
np.divide(absolute_change * 100, first_half_mean, out=percent_change, where=first_half_mean != 0)

print(table(region=region_names,
            first_4_weeks=np.round(first_half_mean, 2),
            last_4_weeks=np.round(last_half_mean, 2),
            abs_change=np.round(absolute_change, 2),
            pct_change=np.round(percent_change, 1)))

store_percent_change = (sales[:, 4:].sum() - sales[:, :4].sum()) / sales[:, :4].sum() * 100
print(f"\nМагазин в целом: {store_percent_change:.1f}%")

[('Bishkek', 125.75, 139.5 , 13.75, 10.9)
 ('Osh',  81.5 ,  89.  ,  7.5 ,  9.2)
 ('Karakol', 153.25, 178.25, 25.  , 16.3)
 ('Tokmok',  61.25,  63.5 ,  2.25,  3.7)
 ('Jalal-Abad', 103.75, 111.5 ,  7.75,  7.5)]

Магазин в целом: 10.7%


**Вывод.** Растут все регионы. Быстрее всех — Karakol (+16.3%, но часть роста даёт выброс) и Bishkek (+10.9%).
Медленнее среднего по магазину (+10.7%) растут Osh (+9.2%), Jalal-Abad (+7.5%) и особенно **Tokmok (+3.7%)**.

## Задание 6. Устойчивость результата

In [7]:
sales_without_outliers = np.where(outlier_mask, np.nan, sales)   # выбросы -> NaN

total_clean = np.nansum(sales_without_outliers, axis=1)
mean_clean = np.nanmean(sales_without_outliers, axis=1)

# сумма по 7 неделям не сопоставима с суммой по 8 — заменяем выброс медианой региона
median_clean = np.nanmedian(sales_without_outliers, axis=1)
sales_imputed = np.where(outlier_mask, median_clean[:, None], sales)
total_imputed = sales_imputed.sum(axis=1)

rank_total_before = (-total_revenue).argsort().argsort() + 1     # 1 = лучший
rank_total_clean = (-total_clean).argsort().argsort() + 1
rank_total_imputed = (-total_imputed).argsort().argsort() + 1
rank_mean_clean = (-mean_clean).argsort().argsort() + 1

print(table(region=region_names,
            total_before=total_revenue,
            total_clean=total_clean,
            total_imputed=total_imputed,
            mean_before=np.round(mean_weekly_sales, 2),
            mean_clean=np.round(mean_clean, 2)))
print()
print(table(region=region_names,
            rank_before=rank_total_before,
            rank_clean=rank_total_clean,
            rank_imputed=rank_total_imputed,
            rank_mean_clean=rank_mean_clean))

# отрыв лидера от второго места
print("\nОтрыв Karakol от Bishkek: до =", total_revenue[2] - total_revenue[0],
      ", после (замена медианой) =", total_imputed[2] - total_imputed[0],
      ", после (исключение, 7 недель) =", total_clean[2] - total_clean[0])

# динамика Karakol без выброса
last_half_mean_clean = np.nanmean(sales_without_outliers[:, 4:], axis=1)
percent_change_clean = (last_half_mean_clean - first_half_mean) / first_half_mean * 100
print("Рост Karakol: было", round(percent_change[2], 1), "% -> без выброса", round(percent_change_clean[2], 1), "%")

[('Bishkek', 1061, 1061., 1061., 132.62, 132.62)
 ('Osh',  682,  682.,  682.,  85.25,  85.25)
 ('Karakol', 1326, 1106., 1264., 165.75, 158.  )
 ('Tokmok',  499,  499.,  499.,  62.38,  62.38)
 ('Jalal-Abad',  861,  861.,  861., 107.62, 107.62)]

[('Bishkek', 2, 2, 2, 2) ('Osh', 4, 4, 4, 4) ('Karakol', 1, 1, 1, 1)
 ('Tokmok', 5, 5, 5, 5) ('Jalal-Abad', 3, 3, 3, 3)]

Отрыв Karakol от Bishkek: до = 265 , после (замена медианой) = 203.0 , после (исключение, 7 недель) = 45.0
Рост Karakol: было 16.3 % -> без выброса 7.2 %


**Вывод.** После исключения выброса сумма Karakol падает с 1326 до 1106 (за 7 недель), среднее — 165.75 → 158.0; у остальных регионов ничего не меняется.
Рейтинг здесь **не изменился**, но отрыв Karakol от Bishkek сократился с 265 до 45 (при исключении) или до 203 (при замене выброса медианой).
В общем случае удаление наблюдений **может изменить рейтинг**: если регионы близки по выручке, а выброс велик, лидер поменяется.
Рейтинг по сумме после исключения некорректен (разное число недель) — сравнивайте по среднему или с заменой выброса.
Рост Karakol без выброса — лишь +7.2% вместо +16.3%.

## Задание 7. Поиск проблемных регионов

In [8]:
# 3 показателя: динамика, нестабильность, частота недельных падений
weekly_drop_share = (np.diff(sales, axis=1) < 0).mean(axis=1)    # доля недель со снижением к предыдущей

flag_slow_growth = percent_change < store_percent_change         # растёт медленнее магазина
flag_unstable = coef_variation > np.median(coef_variation)       # CV выше медианного по регионам
flag_frequent_drops = weekly_drop_share > 0.5                    # падения чаще, чем в половине недель

flags_count = flag_slow_growth.astype(int) + flag_unstable + flag_frequent_drops
is_problem = flags_count >= 2                                    # проблемный: минимум 2 флага из 3

verdict = np.where(is_problem, "Проблемный", np.where(flags_count == 1, "Наблюдать", "Норма"))

print(table(region=region_names,
            share_percent=np.round(revenue_share, 1),
            pct_change=np.round(percent_change, 1),
            cv_percent=np.round(coef_variation, 1),
            drop_share=np.round(weekly_drop_share, 2),
            flags=flags_count,
            verdict=verdict))

[('Bishkek', 24. , 10.9,  6.3, 0.29, 1, 'Наблюдать')
 ('Osh', 15.4,  9.2,  5.5, 0.29, 1, 'Наблюдать')
 ('Karakol', 29.9, 16.3, 13.9, 0.43, 1, 'Наблюдать')
 ('Tokmok', 11.3,  3.7,  4.3, 0.57, 2, 'Проблемный')
 ('Jalal-Abad', 19.4,  7.5,  4.7, 0.29, 1, 'Наблюдать')]


**Критерий:** регион проблемный, если выполнено ≥ 2 из 3 условий:
1. рост ниже среднего по магазину (отстаёт от рынка);
2. CV выше медианного по регионам (нестабильность);
3. недельные падения чаще, чем в половине недель (нет устойчивого тренда).

Показатели относительные (%), поэтому размер региона и низкая выручка сами по себе не делают его проблемным (доля выручки приведена для контекста).

**Результат:** проблемный — **Tokmok** (рост +3.7% при среднем по магазину +10.7%, падения в 57% недель).
Под наблюдением (1 флаг): Karakol и Bishkek (CV выше медианного; у Karakol — из-за выброса), Osh и Jalal-Abad (рост ниже среднего).
Границы условны — это стартовый скрининг, а не окончательный вердикт.

## Задание 8. Записка для руководства

**1. Кто формирует выручку.** Karakol (29.9%) и Bishkek (24.0%) — вместе 53.9%. С Jalal-Abad (19.4%) — 73.3%. Osh (15.4%) и Tokmok (11.3%) — оставшиеся ≈27%.

**2. Где нестабильные продажи.** Karakol (CV 13.9% против 4–6% у остальных), но причина — один скачок на неделе 7. Без него Karakol стабилен.
Tokmok стабилен по CV, но у него частые недельные падения и самый слабый рост (+3.7%).

**3. Что проверить.**
- Karakol, неделя 7 (220 вместо ≈165–170): акция или ошибка ввода — влияет на выручку (≈50 ед.) и на оценку роста региона (+16.3% → +7.2%);
- Tokmok: причины слабого роста и частых снижений;
- Osh и Jalal-Abad: рост ниже среднего по магазину (+9.2% и +7.5% против +10.7%).

**4. Какие данные нужны.** Календарь акций и маркетинга; число заказов и средний чек; число клиентов и размер рынка региона (население, доход);
себестоимость и маржа; остатки и сбои логистики; история продаж > 8 недель (сезонность); источник данных по Karakol, неделя 7.

## Дополнительное задание. Ранжирование регионов

In [9]:
rank_total = (-total_revenue).argsort().argsort() + 1
rank_mean = (-mean_weekly_sales).argsort().argsort() + 1
rank_stability = coef_variation.argsort().argsort() + 1          # 1 = самый стабильный (минимальный CV)
rank_growth = (-percent_change).argsort().argsort() + 1          # рост как косвенный показатель эффективности

print(table(region=region_names,
            by_total=rank_total,
            by_mean=rank_mean,
            by_stability=rank_stability,
            by_growth=rank_growth))

[('Bishkek', 2, 2, 4, 2) ('Osh', 4, 4, 3, 3) ('Karakol', 1, 1, 5, 1)
 ('Tokmok', 5, 5, 1, 5) ('Jalal-Abad', 3, 3, 2, 4)]


**Вывод.**
- Ранги по сумме и по среднему совпадают (одинаковое число недель).
- Karakol — первый по объёму, но **последний по стабильности**; Tokmok — наоборот (5-й по объёму, 1-й по стабильности).
- Объём зависит от размера рынка региона, а стабильность — от колебаний относительно среднего: крупный регион может сильнее зависеть от разовых всплесков.
- Эффективность (отдача на вложения, маржа, продажи на жителя) по этим данным не измерить — нужны данные о затратах и размере рынка.
  Рост даёт лишь косвенную оценку; по нему Karakol тоже лидер, но с поправкой на выброс его преимущество меньше (+7.2%).